In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

drive_path = "/content/drive/MyDrive"
zip_name = "marine-plastic-src.zip"

matches = []

for root, dirs, files in os.walk(drive_path):
    if zip_name in files:
        matches.append(os.path.join(root, zip_name))

if matches:
    print("ZIP FOUND:")
    for path in matches:
        print(path)
else:
    print("ZIP NOT FOUND. Check that it was uploaded to Google Drive.")

In [ ]:
import os
import zipfile
import shutil

zip_path = "/content/drive/MyDrive/marine-plastic-src.zip"

print("ZIP size:", round(os.path.getsize(zip_path) / (1024**2), 2), "MB")

total, used, free = shutil.disk_usage("/content")
print("Colab free space:", round(free / (1024**3), 2), "GB")

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()
    bad_file = z.testzip()

    print("\nNumber of archived files:", len(files))
    print("Archive integrity:", "PASS" if bad_file is None else f"FAILED: {bad_file}")

    unsafe = [
        name for name in files
        if name.startswith("/") or ".." in name.split("/")
    ]
    print("Unsafe paths:", len(unsafe))

    print("\nFirst 20 files:")
    for name in files[:20]:
        print(name)

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/marine-plastic-src.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

project_path = "/content/marine-plastic"

print("Extraction complete.")
print("Project folder exists:", os.path.isdir(project_path))
print("\nTop-level project contents:")

for item in sorted(os.listdir(project_path)):
    print("-", item)

In [ ]:
import os
import subprocess

project_path = "/content/marine-plastic"

result = subprocess.run(
    ["sha256sum", "-c", "MANIFEST.sha256"],
    cwd=project_path,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode == 0:
    print("\nPROJECT INTEGRITY: PASS")
else:
    print("\nPROJECT INTEGRITY: FAILED")
    print(result.stderr)

In [ ]:
import sys
import platform
import shutil
import subprocess

print("Python:", sys.version)
print("Platform:", platform.platform())

gpu = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)

if gpu.returncode == 0:
    print("\nGPU INFORMATION:")
    print(gpu.stdout)
else:
    print("\nNo NVIDIA GPU currently assigned.")

total, used, free = shutil.disk_usage("/content")

print("\nStorage:")
print("Total:", round(total / 1024**3, 2), "GB")
print("Used:", round(used / 1024**3, 2), "GB")
print("Free:", round(free / 1024**3, 2), "GB")

In [ ]:
import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("No GPU assigned yet.")

!nvidia-smi

In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/marine-plastic-src.zip"
project_path = "/content/marine-plastic"

if not os.path.exists(project_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")

print("Project exists:", os.path.isdir(project_path))
print("Project files:", len(os.listdir(project_path)))
print("Top-level folders:", sorted(os.listdir(project_path)))

In [ ]:
from pathlib import Path

project = Path("/content/marine-plastic")

files_to_read = [
    "docs/M3_DATASET_VALIDATION.md",
    "docs/DATASET_CARD.md",
    "requirements/base.txt",
    "requirements/train.txt",
    "configs/data.yaml"
]

for filename in files_to_read:
    path = project / filename
    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    if path.exists():
        print(path.read_text())
    else:
        print("FILE NOT FOUND")

In [ ]:
import importlib.metadata as md
import subprocess
import sys
from pathlib import Path

project = Path("/content/marine-plastic")

packages = [
    "torch", "torchvision", "numpy", "Pillow",
    "PyYAML", "tqdm", "pycocotools",
    "scipy", "matplotlib", "pandas", "imagehash"
]

print("INSTALLED PACKAGES")
for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nFETCH SCRIPT OPTIONS")
result = subprocess.run(
    [sys.executable, str(project / "ml/data/fetch_bepli.py"), "--help"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

print("\nINSPECTION SCRIPT OPTIONS")
result = subprocess.run(
    [sys.executable, str(project / "ml/data/inspect_coco.py"), "--help"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
import os
import sys
import subprocess

project = "/content/marine-plastic"

os.chdir(project)

env = os.environ.copy()
env["PYTHONPATH"] = project

result = subprocess.run(
    [sys.executable, "-m", "ml.data.inspect_coco", "--help"],
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("ERROR / WARNINGS:")
    print(result.stderr)

print("Exit code:", result.returncode)

In [ ]:
%pip install imagehash==4.3.1

In [ ]:
import requests

url = "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz"

response = requests.get(
    url,
    headers={"Range": "bytes=0-0"},
    stream=True,
    allow_redirects=True,
    timeout=30
)

print("HTTP status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Content range:", response.headers.get("Content-Range"))
print("Content length:", response.headers.get("Content-Length"))
print("Final URL:", response.url)

response.close()

In [ ]:
import os
os.chdir("/content/marine-plastic")

!python -m ml.data.fetch_bepli \
  --url "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz" \
  --dest /content/bepli_v2 \
  --min-free-gb 25

In [ ]:
from pathlib import Path

root = Path("/content/bepli_v2/raw")

print("Dataset root exists:", root.exists())
print("\nTop-level contents:")

for item in sorted(root.iterdir()):
    if item.is_dir():
        children = list(item.iterdir())
        print(f"[DIR]  {item.name}/  ({len(children)} immediate entries)")
    else:
        print(f"[FILE] {item.name}  ({item.stat().st_size:,} bytes)")


In [ ]:
from pathlib import Path

root = Path("/content/bepli_v2/raw/BePLi_dataset_v2")

print("Dataset directory exists:", root.exists())
print("\nContents:")

for item in sorted(root.iterdir()):
    if item.is_dir():
        print(f"[DIR]  {item.name}/ ({len(list(item.iterdir()))} immediate entries)")
    else:
        print(f"[FILE] {item.name} ({item.stat().st_size:,} bytes)")

In [ ]:
from pathlib import Path

root = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")

print("Contents of plastic_coco:\n")

for item in sorted(root.iterdir()):
    if item.is_dir():
        print(f"[DIR]  {item.name}/ ({len(list(item.iterdir()))} immediate entries)")
    else:
        print(f"[FILE] {item.name} ({item.stat().st_size:,} bytes)")

In [ ]:
from pathlib import Path

base = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")

for folder_name in ["annotation", "images"]:
    folder = base / folder_name
    print(f"\n{'='*50}\n{folder_name.upper()}\n{'='*50}")

    for item in sorted(folder.iterdir()):
        if item.is_dir():
            print(f"[DIR]  {item.name}/ ({len(list(item.iterdir()))} immediate entries)")
        else:
            print(f"[FILE] {item.name} ({item.stat().st_size:,} bytes)")

In [ ]:
from pathlib import Path
import subprocess
import sys

base = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")

print("========== DATASET README ==========\n")
print((base / "README").read_text(errors="replace"))

print("\n========== INSPECTOR OPTIONS ==========\n")
project = Path("/content/marine-plastic")
result = subprocess.run(
    [sys.executable, "-m", "ml.data.inspect_coco", "--help"],
    cwd=project,
    capture_output=True,
    text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

In [ ]:
from pathlib import Path

p = Path("/content/marine-plastic/ml/data/inspect_coco.py")
lines = p.read_text().splitlines()

terms = ["file_name", "images", "args.root", "root /", "ann_path"]

for term in terms:
    print(f"\n{'='*20} {term} {'='*20}")
    matches = [i for i, line in enumerate(lines) if term in line]
    for i in matches[:5]:
        start, end = max(0, i-3), min(len(lines), i+5)
        for j in range(start, end):
            print(f"{j+1}: {lines[j]}")
        print("---")

In [ ]:
from pathlib import Path

p = Path("/content/marine-plastic/ml/data/inspect_coco.py")
lines = p.read_text().splitlines()

print("========== IMAGE RESOLUTION ==========")
for i in range(14, 40):
    print(f"{i+1}: {lines[i]}")

print("\n========== MAIN / ANNOTATION DISCOVERY ==========")
for i in range(145, len(lines)):
    print(f"{i+1}: {lines[i]}")

In [ ]:
from pathlib import Path

p = Path("/content/marine-plastic/ml/data/coco_utils.py")
lines = p.read_text().splitlines()

for i, line in enumerate(lines):
    if line.startswith("def build_image_index") or line.startswith("def resolve_image"):
        print(f"\n========== {line.strip()} ==========")
        for j in range(i, min(i + 45, len(lines))):
            print(f"{j+1}: {lines[j]}")

In [ ]:
import json
from pathlib import Path

base = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
ann_dir = base / "annotation"

for filename in [
    "all_plastic_coco.json",
    "train.json",
    "val.json",
    "test.json"
]:
    data = json.loads((ann_dir / filename).read_text())

    print(f"\n========== {filename} ==========")
    print("Images:", len(data["images"]))
    print("Annotations:", len(data["annotations"]))
    print("Categories:", len(data["categories"]))
    print("Sample image filenames:")

    for im in data["images"][:5]:
        print(" ", im["file_name"])

In [ ]:
from pathlib import Path
import subprocess
import sys

project = Path("/content/marine-plastic")
base = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
ann = base / "annotation"
images = base / "images"

jobs = [
    (
        "COMPLETE DATASET",
        images / "original_images",
        ann / "all_plastic_coco.json",
        "all_plastic",
        ["--expect-images", "3722",
         "--expect-annotations", "118572",
         "--expect-classes", "13"]
    ),
    ("TRAIN", images / "train", ann / "train.json", "train", []),
    ("VALIDATION", images / "val", ann / "val.json", "val", []),
    ("TEST", images / "test", ann / "test.json", "test", [])
]

for name, image_root, annotation, output, expected in jobs:
    print(f"\n{'='*65}")
    print(f"INSPECTING: {name}")
    print(f"{'='*65}")

    command = [
        sys.executable, "-m", "ml.data.inspect_coco",
        "--root", str(image_root),
        "--ann", str(annotation),
        "--out", f"reports/m3/{output}",
        "--sample-decode", "300",
        *expected
    ]

    result = subprocess.run(
        command,
        cwd=project,
        capture_output=True,
        text=True
    )

    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

    if result.returncode != 0:
        print(f"INSPECTION FAILED: {name}")
        break
    else:
        print(f"INSPECTION COMPLETED: {name}")

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

project = Path("/content/marine-plastic")

# 1. Read the four detailed inspection reports
for name in ["all_plastic", "train", "val", "test"]:
    path = project / f"reports/m3/{name}.json"
    report = json.loads(path.read_text())

    print(f"\n{'='*60}\n{name.upper()} DETAILS\n{'='*60}")

    for key in [
        "duplicate_image_ids",
        "duplicate_file_names",
        "images_missing_on_disk",
        "sampled_decode_failures",
        "rle_decode_check",
        "segmentation_types",
        "iscrowd_count",
        "images_without_annotations",
        "class_imbalance_ratio_max_over_min",
        "per_class_instances",
        "problems"
    ]:
        print(f"{key}: {report.get(key, 'NOT FOUND')}")

# 2. Check overlap between the three official splits
base = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
ann = base / "annotation"
images = base / "images"

command = [
    sys.executable, "-m", "ml.data.inspect_coco",
    "--root", str(images),
    "--ann", str(ann / "train.json"),
    "--ann", str(ann / "val.json"),
    "--ann", str(ann / "test.json"),
    "--out", "reports/m3/official_split_overlap",
    "--sample-decode", "50"
]

print("\n========== OFFICIAL SPLIT OVERLAP CHECK ==========")
result = subprocess.run(
    command, cwd=project, capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

In [ ]:
from pathlib import Path
import json

project = Path("/content/marine-plastic")

for name in ["all_plastic", "train", "val", "test", "official_split_overlap"]:
    json_path = project / f"reports/m3/{name}.json"
    md_path = project / f"reports/m3/{name}.md"

    print(f"\n{'='*70}")
    print(f"REPORT: {name.upper()}")
    print(f"{'='*70}")

    if json_path.exists():
        data = json.loads(json_path.read_text())
        print("\nJSON TOP-LEVEL KEYS:")
        print(list(data.keys()))

        print("\nFULL JSON REPORT:")
        print(json.dumps(data, indent=2)[:12000])

    if md_path.exists():
        print("\nMARKDOWN REPORT:")
        print(md_path.read_text()[:12000])

In [ ]:
import json
from pathlib import Path
from itertools import combinations

base = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
ann = base / "annotation"

def load_split(name):
    data = json.loads((ann / f"{name}.json").read_text())
    return {
        "filenames": {img["file_name"] for img in data["images"]},
        "ids": {img["id"] for img in data["images"]},
        "categories": {
            c["id"]: c["name"] for c in data["categories"]
        }
    }

splits = {
    name: load_split(name)
    for name in ["train", "val", "test"]
}

all_data = json.loads((ann / "all_plastic_coco.json").read_text())
all_filenames = {img["file_name"] for img in all_data["images"]}
all_ids = {img["id"] for img in all_data["images"]}

print("========== CROSS-SPLIT OVERLAP ==========")

for a, b in combinations(splits, 2):
    filename_overlap = (
        splits[a]["filenames"] & splits[b]["filenames"]
    )
    id_overlap = splits[a]["ids"] & splits[b]["ids"]

    print(f"\n{a.upper()} vs {b.upper()}")
    print("Shared filenames:", len(filename_overlap))
    print("Shared image IDs:", len(id_overlap))

    if filename_overlap:
        print("Examples:", list(filename_overlap)[:10])
    if id_overlap:
        print("ID examples:", list(id_overlap)[:10])

print("\n========== PARTITION CHECK ==========")

combined_names = set().union(
    *(s["filenames"] for s in splits.values())
)
combined_ids = set().union(
    *(s["ids"] for s in splits.values())
)

print("Split filename union equals complete dataset:",
      combined_names == all_filenames)
print("Split image-ID union equals complete dataset:",
      combined_ids == all_ids)

print("Complete dataset filenames:", len(all_filenames))
print("Combined split filenames:", len(combined_names))
print("Complete dataset image IDs:", len(all_ids))
print("Combined split image IDs:", len(combined_ids))

print("\n========== CATEGORY CONSISTENCY ==========")

for name, split in splits.items():
    print(name, "categories match complete dataset:",
          split["categories"] == {
              c["id"]: c["name"] for c in all_data["categories"]
          })

In [ ]:
import json
import time
import numpy as np
from pathlib import Path
from pycocotools import mask as mask_utils

base = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
ann_dir = base / "annotation"

total = 0
failed = []
size_mismatches = []
area_mismatches = []
empty_masks = []

start = time.time()

for split in ["train", "val", "test"]:
    data = json.loads((ann_dir / f"{split}.json").read_text())

    image_info = {
        img["id"]: (img["height"], img["width"])
        for img in data["images"]
    }

    split_total = 0
    print(f"\nChecking {split.upper()}...")

    for annotation in data["annotations"]:
        total += 1
        split_total += 1
        ann_id = annotation["id"]

        try:
            segmentation = annotation["segmentation"]
            decoded = mask_utils.decode(segmentation)

            expected_h, expected_w = image_info[annotation["image_id"]]

            if decoded.shape[:2] != (expected_h, expected_w):
                size_mismatches.append({
                    "split": split,
                    "annotation_id": ann_id,
                    "decoded_shape": decoded.shape,
                    "expected_shape": (expected_h, expected_w)
                })
                continue

            pixel_area = int(np.count_nonzero(decoded))

            if pixel_area == 0:
                empty_masks.append((split, ann_id))

            if pixel_area != int(annotation["area"]):
                area_mismatches.append({
                    "split": split,
                    "annotation_id": ann_id,
                    "decoded_area": pixel_area,
                    "reported_area": annotation["area"]
                })

        except Exception as e:
            failed.append({
                "split": split,
                "annotation_id": ann_id,
                "error": str(e)
            })

    print(f"Annotations checked: {split_total:,}")

print("\n========== FULL RLE VALIDATION ==========")
print("Total annotations checked:", f"{total:,}")
print("Decode failures:", len(failed))
print("Mask dimension mismatches:", len(size_mismatches))
print("Mask area mismatches:", len(area_mismatches))
print("Empty masks:", len(empty_masks))
print("Time taken:", round(time.time() - start, 2), "seconds")

if failed:
    print("\nDecode failure examples:", failed[:5])

if size_mismatches:
    print("\nDimension mismatch examples:", size_mismatches[:5])

if area_mismatches:
    print("\nArea mismatch examples:", area_mismatches[:5])

if empty_masks:
    print("\nEmpty mask examples:", empty_masks[:5])

In [ ]:
import json
import random
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
from pycocotools import mask as mask_utils

base = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
ann_dir = base / "annotation"

rng = random.Random(2026)

for split in ["train", "val", "test"]:

    data = json.loads((ann_dir / f"{split}.json").read_text())

    image_dir = base / "images" / split

    categories = {
        c["id"]: c["name"]
        for c in data["categories"]
    }

    annotations_by_image = {}

    for a in data["annotations"]:
        annotations_by_image.setdefault(
            a["image_id"], []
        ).append(a)

    images = data["images"]
    selected = rng.sample(images, 10)

    print(f"\n{'='*55}")
    print(f"{split.upper()} — 10 RANDOM IMAGES")
    print(f"{'='*55}")

    fig, axes = plt.subplots(2, 5, figsize=(22, 10))
    axes = axes.flatten()

    for ax, img_info in zip(axes, selected):

        image_path = image_dir / img_info["file_name"]

        image = np.array(
            Image.open(image_path).convert("RGB")
        )

        ax.imshow(image)

        anns = annotations_by_image.get(
            img_info["id"], []
        )

        present_classes = set()

        for a in anns:
            mask = mask_utils.decode(a["segmentation"])

            color = plt.get_cmap("tab20")(
                (a["category_id"] - 1) % 20
            )[:3]

            overlay = np.zeros(
                (*mask.shape, 4),
                dtype=np.float32
            )

            overlay[..., :3] = color
            overlay[..., 3] = mask * 0.35

            ax.imshow(overlay)

            present_classes.add(
                categories[a["category_id"]]
            )

        ax.set_title(
            f"{img_info['file_name']}\n"
            f"Objects: {len(anns)}",
            fontsize=9
        )

        ax.axis("off")

    plt.suptitle(
        f"BePLi v2 — {split.upper()} — Segmentation Review",
        fontsize=15
    )

    plt.tight_layout()
    plt.show()

    print("Classes represented in this batch:")
    print(", ".join(sorted({
        categories[a["category_id"]]
        for img in selected
        for a in annotations_by_image.get(img["id"], [])
    })))

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

base = Path(
    "/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco"
)
ann_dir = base / "annotation"

for split in ["train", "val", "test"]:

    data = json.loads(
        (ann_dir / f"{split}.json").read_text()
    )

    annotated_ids = {
        a["image_id"] for a in data["annotations"]
    }

    empty_images = [
        img for img in data["images"]
        if img["id"] not in annotated_ids
    ]

    print(f"\n{split.upper()}: {len(empty_images)} images")

    if not empty_images:
        continue

    fig, axes = plt.subplots(
        1, len(empty_images),
        figsize=(6 * len(empty_images), 5)
    )

    if len(empty_images) == 1:
        axes = [axes]

    for ax, img_info in zip(axes, empty_images):

        image_path = (
            base / "images" / split / img_info["file_name"]
        )

        image = Image.open(image_path).convert("RGB")

        ax.imshow(image)
        ax.set_title(img_info["file_name"])
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path

dataset = Path(
    "/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco"
)

print("Dataset exists:", dataset.exists())
print("Annotations exist:",
      (dataset / "annotation/train.json").exists())

print("Project exists:",
      Path("/content/marine-plastic").exists())

In [ ]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

print("Google Drive mounted:", drive.exists())
print("Project ZIP exists:",
      (drive / "marine-plastic-src.zip").exists())

In [ ]:
from pathlib import Path

archive = Path("/content/drive/MyDrive/106157.tar.gz")

print("Dataset archive in Drive:", archive.exists())

In [ ]:
from pathlib import Path
import json

base = Path(
    "/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco"
)

ann_dir = base / "annotation"

print("Dataset folder exists:", base.exists())

for name in ["train.json", "val.json", "test.json"]:
    path = ann_dir / name
    print(f"{name} exists:", path.exists())

    if path.exists():
        data = json.loads(path.read_text())
        print(
            f"  Images: {len(data['images']):,} | "
            f"Annotations: {len(data['annotations']):,}"
        )

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

base = Path(
    "/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco"
)
ann_dir = base / "annotation"

for split in ["train", "val", "test"]:

    data = json.loads(
        (ann_dir / f"{split}.json").read_text()
    )

    annotated_ids = {
        a["image_id"] for a in data["annotations"]
    }

    empty_images = [
        img for img in data["images"]
        if img["id"] not in annotated_ids
    ]

    for img_info in empty_images:

        image_path = (
            base / "images" / split / img_info["file_name"]
        )

        image = Image.open(image_path).convert("RGB")

        plt.figure(figsize=(12, 7))
        plt.imshow(image)
        plt.title(f"{split.upper()} — {img_info['file_name']}")
        plt.axis("off")
        plt.show()

In [ ]:
import json
import numpy as np
from pathlib import Path
from collections import Counter

base = Path(
    "/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco"
)
ann_dir = base / "annotation"

for split in ["train", "val", "test"]:

    data = json.loads(
        (ann_dir / f"{split}.json").read_text()
    )

    image_info = {
        img["id"]: img for img in data["images"]
    }

    stats = {
        image_id: {
            "filename": img["file_name"],
            "width": img["width"],
            "height": img["height"],
            "objects": 0,
            "area": 0
        }
        for image_id, img in image_info.items()
    }

    for ann in data["annotations"]:
        image_id = ann["image_id"]

        if image_id in stats:
            stats[image_id]["objects"] += 1
            stats[image_id]["area"] += ann.get("area", 0)

    rows = list(stats.values())

    counts = [r["objects"] for r in rows]
    area_fractions = [
        r["area"] / (r["width"] * r["height"])
        for r in rows
    ]

    print("\n" + "=" * 65)
    print(f"{split.upper()} — LITTER DISTRIBUTION")
    print("=" * 65)

    print("Total images:", len(rows))
    print("Images with zero annotations:", sum(c == 0 for c in counts))
    print("Images with 1–5 objects:", sum(1 <= c <= 5 for c in counts))
    print("Images with 6–20 objects:", sum(6 <= c <= 20 for c in counts))
    print("Images with 21–50 objects:", sum(21 <= c <= 50 for c in counts))
    print("Images with more than 50 objects:", sum(c > 50 for c in counts))

    print("\nObjects per image:")
    print("  Median:", round(float(np.median(counts)), 2))
    print("  Mean:", round(float(np.mean(counts)), 2))
    print("  90th percentile:", round(float(np.percentile(counts, 90)), 2))
    print("  Maximum:", max(counts))

    print("\nAnnotated area as percentage of image area:")
    print("  Median:", round(float(np.median(area_fractions)) * 100, 3), "%")
    print("  90th percentile:", round(float(np.percentile(area_fractions, 90)) * 100, 3), "%")
    print("  Maximum:", round(max(area_fractions) * 100, 3), "%")

    print("\nTOP 15 IMAGES BY OBJECT COUNT")
    for r in sorted(rows, key=lambda x: x["objects"], reverse=True)[:15]:
        print(
            f"{r['filename']:15} "
            f"objects={r['objects']:3} "
            f"annotated_area={r['area']:,} px"
        )

    print("\nTOP 15 IMAGES BY ANNOTATED AREA")
    for r in sorted(rows, key=lambda x: x["area"], reverse=True)[:15]:
        print(
            f"{r['filename']:15} "
            f"area={r['area']:,} px "
            f"objects={r['objects']:3}"
        )

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path
from PIL import Image
from pycocotools import mask as mask_utils

base = Path(
    "/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco"
)

ann_dir = base / "annotation"

for split in ["train", "val", "test"]:

    data = json.loads(
        (ann_dir / f"{split}.json").read_text()
    )

    images = {img["id"]: img for img in data["images"]}

    area_by_image = {}

    for ann in data["annotations"]:
        image_id = ann["image_id"]
        area_by_image[image_id] = (
            area_by_image.get(image_id, 0)
            + ann.get("area", 0)
        )

    top_images = sorted(
        area_by_image,
        key=area_by_image.get,
        reverse=True
    )[:5]

    print(f"\n{split.upper()} — TOP 5 BY ANNOTATED AREA")

    fig, axes = plt.subplots(
        1, 5, figsize=(24, 7)
    )

    for ax, image_id in zip(axes, top_images):

        info = images[image_id]

        image_path = (
            base / "images" / split / info["file_name"]
        )

        image = np.array(
            Image.open(image_path).convert("RGB")
        )

        ax.imshow(image)

        annotations = [
            ann for ann in data["annotations"]
            if ann["image_id"] == image_id
        ]

        combined_mask = np.zeros(
            (info["height"], info["width"]),
            dtype=bool
        )

        for ann in annotations:
            seg = ann["segmentation"]

            if isinstance(seg, dict):
                mask = mask_utils.decode(seg)

                if mask.ndim == 3:
                    mask = np.any(mask, axis=2)

                combined_mask |= mask.astype(bool)

        ax.imshow(
            np.ma.masked_where(
                ~combined_mask, combined_mask
            ),
            alpha=0.5,
            cmap="autumn"
        )

        coverage = (
            area_by_image[image_id]
            / (info["width"] * info["height"])
        ) * 100

        ax.set_title(
            f"{info['file_name']}\n"
            f"Objects: {len(annotations)}\n"
            f"Annotated area: {coverage:.1f}%"
        )

        ax.axis("off")

    plt.suptitle(
        f"{split.upper()} — Most heavily annotated images"
    )
    plt.tight_layout()
    plt.show()

In [ ]:
!find /content/bepli_v2/raw -maxdepth 4 -type d | head -50
!ls /content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco/images | head

In [ ]:
!ls -lah /content
!find /content -maxdepth 4 -type d -iname '*bepli*'

In [ ]:
!ls -lah /content/drive/MyDrive 2>/dev/null | head -30

In [ ]:
!ls -ld /content/drive /content/drive/MyDrive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!find /content/drive/MyDrive -maxdepth 5 -type d -iname '*bepli*'

In [ ]:
!ls -lah /content/drive/MyDrive

In [ ]:
!find /content/drive/MyDrive -type f \( -iname '*bepli*' -o -iname '*.tar.gz' \)

In [ ]:
# ===== STEP 1: restore BePLi v2 on Colab (download + safe extract + verify). Safe to re-run. =====
import glob, hashlib, json, os, shutil, subprocess, tarfile, time

URL  = "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz"
BASE = "/content/bepli_v2"; DEST = f"{BASE}/raw"; DONE = f"{BASE}/EXTRACT_DONE"
PUBLISHED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}   # (images, annotations)
os.makedirs(BASE, exist_ok=True)

free = shutil.disk_usage("/content").free / 1e9
print(f"Free disk: {free:.1f} GB")
assert free > 16, "Not enough free disk (need ~16 GB). Runtime > Disconnect and delete runtime, then run again."

class HashReader:                                   # hashes every byte that streams through
    def __init__(self, raw): self.raw, self.h, self.n = raw, hashlib.sha256(), 0
    def read(self, size=-1):
        b = self.raw.read(size); self.h.update(b); self.n += len(b); return b

if os.path.exists(DONE):
    print("Dataset already extracted in this session - skipping download.")
else:
    shutil.rmtree(DEST, ignore_errors=True); os.makedirs(DEST)          # remove any half-finished extraction from an earlier attempt
    print("Downloading + extracting (about 8 GB; no archive file is stored). Do not close the tab...")
    t0 = time.time(); root = os.path.realpath(DEST)
    proc = subprocess.Popen(["curl", "-fsSL", "--retry", "3", URL], stdout=subprocess.PIPE)
    rd = HashReader(proc.stdout); n = 0
    kw = {"filter": "data"} if hasattr(tarfile, "data_filter") else {}
    with tarfile.open(fileobj=rd, mode="r|gz") as tf:
        for m in tf:
            tgt = os.path.realpath(os.path.join(root, m.name))
            if not tgt.startswith(root) or m.issym() or m.islnk(): raise RuntimeError(f"unsafe archive member: {m.name}")
            tf.extract(m, root, **kw); n += 1
            if n % 1000 == 0: print(f"  {n} files, {rd.n/1e9:.2f} GB read, {(time.time()-t0)/60:.1f} min", flush=True)
    while rd.read(1 << 20): pass
    proc.stdout.close(); rc = proc.wait()
    if rc != 0: raise RuntimeError(f"download failed (curl exit {rc}). Run this cell again.")
    json.dump({"url": URL, "files": n, "bytes": rd.n, "sha256": rd.h.hexdigest(), "minutes": round((time.time()-t0)/60, 1)}, open(f"{BASE}/DOWNLOAD_RECORD.json", "w"), indent=1)
    open(DONE, "w").write("ok"); print(f"Extracted {n} files, {rd.n/1e9:.2f} GB in {(time.time()-t0)/60:.1f} min")

# ---- locate the annotation folder (train/val/test only; all_plastic_coco.json is deliberately NOT used) ----
dirs = {s: sorted({os.path.dirname(p) for p in glob.glob(f"{DEST}/**/{s}.json", recursive=True)}) for s in PUBLISHED}
common = set(dirs["train"]) & set(dirs["val"]) & set(dirs["test"])
assert len(common) == 1, f"Could not find exactly one folder holding train/val/test.json: {dirs}"
ANN = common.pop(); IMG = os.path.join(os.path.dirname(ANN), "images")
print(f"\nAnnotation folder: {ANN}\nImage folder     : {IMG}/{{train,val,test}}")
ok_all = True
for s, (ni, na) in PUBLISHED.items():
    d = json.load(open(f"{ANN}/{s}.json")); files = len([f for f in os.listdir(f"{IMG}/{s}") if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    ok = (len(d["images"]), len(d["annotations"]), files) == (ni, na, ni); ok_all &= ok
    print(f"  {s:<5} JSON images {len(d['images']):>5} | annotations {len(d['annotations']):>6} | image files {files:>5}   (published {ni} / {na})  {'PASS' if ok else 'DIFFERENT'}")
print("\nStep 1 RESULT:", "ALL COUNTS MATCH THE PUBLISHED VALUES" if ok_all else "SOMETHING DIFFERS - send this whole output back")
os.system("pip -q install pycocotools==2.0.8")
print("pycocotools installed.")

In [ ]:
# ===== STEP 1 (v2): resumable download -> verified archive -> extract -> verify. Safe to re-run. =====
import glob, json, os, shutil, subprocess, tarfile, time

URL  = "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz"
BASE = "/content/bepli_v2"; DEST = f"{BASE}/raw"; DONE = f"{BASE}/EXTRACT_DONE"; ARCH = f"{BASE}/bepli_v2.tar.gz"
DRIVE_DIR = "/content/drive/MyDrive/marine-plastic-data"; ARCH_DRIVE = f"{DRIVE_DIR}/bepli_v2.tar.gz"
PUBLISHED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}   # (images, annotations)
os.makedirs(BASE, exist_ok=True)

try:                                                  # Drive is used only to keep ONE verified copy of the archive (so it is never downloaded twice)
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
    HAVE_DRIVE = os.path.isdir("/content/drive/MyDrive")
except Exception:
    HAVE_DRIVE = False
print("Google Drive available:", HAVE_DRIVE)

free = shutil.disk_usage("/content").free / 1e9
print(f"Free disk: {free:.1f} GB")
assert free > 20, "Not enough free disk (need ~20 GB). Runtime > Disconnect and delete runtime, then run again."

def expected_size():
    out = subprocess.run(["curl", "-sIL", URL], capture_output=True, text=True).stdout
    sizes = [int(l.split(":")[1]) for l in out.splitlines() if l.lower().startswith("content-length:")]
    return sizes[-1] if sizes else None

def gz_ok(path): return subprocess.run(["gzip", "-t", path], capture_output=True).returncode == 0

def download(expected):
    """Resumes after every dropped connection (curl -C -). Up to 25 attempts."""
    for attempt in range(1, 26):
        have = os.path.getsize(ARCH) if os.path.exists(ARCH) else 0
        if expected and have == expected: return True
        if expected and have > expected: os.remove(ARCH); have = 0
        print(f"Download attempt {attempt}: have {have/1e9:.2f} GB" + (f" of {expected/1e9:.2f} GB" if expected else ""), flush=True)
        p = subprocess.Popen(["curl", "-sS", "-L", "-C", "-", "--retry", "3", "--retry-delay", "5", "-o", ARCH, URL], stderr=subprocess.PIPE, text=True)
        last = time.time()
        while p.poll() is None:
            time.sleep(2)
            if time.time() - last >= 30:
                print(f"   {os.path.getsize(ARCH)/1e9:.2f} GB downloaded", flush=True); last = time.time()
        err = p.stderr.read().strip()
        if p.returncode == 0 and (not expected or os.path.getsize(ARCH) == expected): return True
        if p.returncode == 33:
            print("   server cannot resume -> restarting from zero"); os.path.exists(ARCH) and os.remove(ARCH)
        else:
            print(f"   interrupted (curl exit {p.returncode}) {err[:100]} -> resuming", flush=True)
        time.sleep(5)
    return False

if os.path.exists(DONE):
    print("Dataset already extracted in this session - skipping download.")
else:
    arch = None
    if HAVE_DRIVE and os.path.exists(ARCH_DRIVE):
        print("Found an archive on Drive - checking it (gzip -t, ~2 min)...")
        if gz_ok(ARCH_DRIVE): arch = ARCH_DRIVE; print("Using the verified archive from Drive: no download needed.")
        else: print("Drive archive is incomplete/corrupt - ignoring it.")
    if arch is None:
        exp = expected_size(); print(f"Server reports archive size: {exp/1e9:.2f} GB" if exp else "Server did not report the size.")
        for full_try in (1, 2):
            if not download(exp): raise RuntimeError("Download did not complete after 25 attempts. Wait a few minutes and run this cell again (it resumes).")
            print("Checking archive integrity (gzip -t, ~2 min)...", flush=True)
            if gz_ok(ARCH): arch = ARCH; print("Archive is complete and intact."); break
            print("Archive failed the integrity check -> deleting it and downloading again."); os.remove(ARCH)
        assert arch, "Archive failed the integrity check twice. Send me this whole output."
    shutil.rmtree(DEST, ignore_errors=True); os.makedirs(DEST)             # discard any partial extraction from earlier attempts
    print("Extracting from the verified archive...", flush=True); t0 = time.time(); root = os.path.realpath(DEST); n = 0
    kw = {"filter": "data"} if hasattr(tarfile, "data_filter") else {}
    with tarfile.open(arch, "r:gz") as tf:
        for m in tf:
            tgt = os.path.realpath(os.path.join(root, m.name))
            if not tgt.startswith(root) or m.issym() or m.islnk(): raise RuntimeError(f"unsafe archive member: {m.name}")
            tf.extract(m, root, **kw); n += 1
            if n % 1000 == 0: print(f"  {n} files, {(time.time()-t0)/60:.1f} min", flush=True)
    json.dump({"url": URL, "files": n, "archive": arch, "archive_bytes": os.path.getsize(arch)}, open(f"{BASE}/DOWNLOAD_RECORD.json", "w"), indent=1)
    open(DONE, "w").write("ok"); print(f"Extracted {n} files in {(time.time()-t0)/60:.1f} min")

dirs = {s: sorted({os.path.dirname(p) for p in glob.glob(f"{DEST}/**/{s}.json", recursive=True)}) for s in PUBLISHED}
common = set(dirs["train"]) & set(dirs["val"]) & set(dirs["test"])
assert len(common) == 1, f"Could not find exactly one folder holding train/val/test.json: {dirs}"
ANN = common.pop(); IMG = os.path.join(os.path.dirname(ANN), "images")
print(f"\nAnnotation folder: {ANN}\nImage folder     : {IMG}/{{train,val,test}}")
ok_all = True
for s, (ni, na) in PUBLISHED.items():
    d = json.load(open(f"{ANN}/{s}.json")); files = len([f for f in os.listdir(f"{IMG}/{s}") if f.lower().endswith((".png", ".jpg", ".jpeg"))])
    ok = (len(d["images"]), len(d["annotations"]), files) == (ni, na, ni); ok_all &= ok
    print(f"  {s:<5} JSON images {len(d['images']):>5} | annotations {len(d['annotations']):>6} | image files {files:>5}   (published {ni} / {na})  {'PASS' if ok else 'DIFFERENT'}")
print("\nStep 1 RESULT:", "ALL COUNTS MATCH THE PUBLISHED VALUES" if ok_all else "SOMETHING DIFFERS - send this whole output back")

if ok_all and HAVE_DRIVE and os.path.exists(ARCH) and not os.path.exists(ARCH_DRIVE):      # keep one verified copy so a runtime reset never costs another download
    need = os.path.getsize(ARCH) + 2e9; dfree = shutil.disk_usage("/content/drive").free
    if dfree > need:
        os.makedirs(DRIVE_DIR, exist_ok=True); print("Saving the verified archive to Drive (one-time, a few minutes)...", flush=True)
        shutil.copyfile(ARCH, ARCH_DRIVE + ".part"); os.replace(ARCH_DRIVE + ".part", ARCH_DRIVE); print("Saved:", ARCH_DRIVE)
    else:
        print(f"Drive has only {dfree/1e9:.1f} GB free - not saving the archive there (it stays on this runtime only).")
os.system("pip -q install pycocotools==2.0.8"); print("pycocotools installed.")

In [ ]:
import os, json, shutil
from pathlib import Path

ROOT = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
EXPECTED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}
SAFE = Path("/content/bepli_safe"); SAFE.mkdir(exist_ok=True)

if not ROOT.exists():
    raise SystemExit("STOP: extracted dataset not found in this runtime. Share this message.")

all_ok, manifest = True, {}
for s, (ni, na) in EXPECTED.items():
    cands = [p for p in sorted((ROOT / "annotation").glob("*.json")) if s in p.name.lower()]
    if not cands:
        print(f"STOP: no annotation JSON for {s}"); all_ok = False; continue
    d = json.load(open(cands[0]))
    n_files = sum(1 for _ in (ROOT / "images" / s).iterdir())
    ok = (len(d["images"]), len(d["annotations"])) == (ni, na) and n_files >= ni
    all_ok &= ok
    manifest[s] = {"annotation": cands[0].name, "images_json": len(d["images"]),
                   "anns_json": len(d["annotations"]), "image_files_on_disk": n_files}
    shutil.copy(cands[0], SAFE / cands[0].name)
    print(f"{s}: json images={len(d['images'])} anns={len(d['annotations'])} files_on_disk={n_files} -> {'OK' if ok else 'MISMATCH'}")

json.dump(manifest, open(SAFE / "manifest.json", "w"), indent=2)
os.system("df -h /content | tail -1")

# remove a fake, unmounted /content/drive folder left by the failed mount (local and ephemeral only)
if Path("/content/drive").exists() and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)
    print("Removed empty unmounted /content/drive stub.")
print("Drive mounted:", os.path.ismount("/content/drive"))

print("\nDATASET VERIFIED IN RUNTIME." if all_ok else "\nSTOP: dataset check failed.")
print("Local annotation backup:", SAFE)

In [ ]:
import os, sys, json, time, shutil, subprocess, tarfile
from pathlib import Path

URL = "https://www.seanoe.org/data/00858/96963/data/106157.tar.gz"
BASE = Path("/content/bepli_v2"); RAW = BASE / "raw"; ARCH = BASE / "bepli_v2.tar.gz"
EXPECTED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}
BASE.mkdir(parents=True, exist_ok=True); RAW.mkdir(parents=True, exist_ok=True)
t0 = time.time()

def find_root():
    for p in RAW.rglob("plastic_coco"):
        if (p / "annotation").is_dir() and (p / "images").is_dir():
            return p
    return None

ROOT = find_root()
if ROOT is None:
    free = shutil.disk_usage("/content").free / 1e9
    print(f"Free disk: {free:.1f} GB")
    if free < 20:
        raise SystemExit("STOP: less than 20 GB free in /content. Share this message.")
    if not ARCH.exists() or ARCH.stat().st_size < 7e9:
        print("Downloading ...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-c", "--tries=5", "--timeout=60", "-O", str(ARCH), URL])
        if rc != 0:
            raise SystemExit(f"STOP: download failed (wget exit {rc}). Share this message.")
    print(f"Archive size: {ARCH.stat().st_size/1e9:.2f} GB | {time.time()-t0:.0f}s")
    print("Testing archive integrity (gzip -t) ...")
    if subprocess.call(["gzip", "-t", str(ARCH)]) != 0:
        raise SystemExit("STOP: archive corrupt. Delete /content/bepli_v2/bepli_v2.tar.gz and rerun this cell.")
    print("Extracting ...")
    if subprocess.call(["tar", "-xzf", str(ARCH), "-C", str(RAW)]) != 0:
        raise SystemExit("STOP: extraction failed. Share this message.")
    ROOT = find_root()
    if ROOT is None:
        raise SystemExit("STOP: plastic_coco folder not found after extraction. Share the output of: !find /content/bepli_v2/raw -maxdepth 3 -type d")
    os.remove(ARCH)  # free local disk; the dataset stays extracted

print("Dataset root:", ROOT)
all_ok, manifest = True, {}
SAFE = Path("/content/bepli_safe"); SAFE.mkdir(exist_ok=True)
for s, (ni, na) in EXPECTED.items():
    cands = [p for p in sorted((ROOT / "annotation").glob("*.json")) if s in p.name.lower()]
    if not cands:
        print(f"STOP: no annotation JSON for {s}"); all_ok = False; continue
    d = json.load(open(cands[0]))
    n_files = sum(1 for _ in (ROOT / "images" / s).iterdir())
    ok = (len(d["images"]), len(d["annotations"])) == (ni, na) and n_files >= ni
    all_ok &= ok
    manifest[s] = {"annotation": cands[0].name, "images_json": len(d["images"]),
                   "anns_json": len(d["annotations"]), "image_files_on_disk": n_files}
    shutil.copy(cands[0], SAFE / cands[0].name)
    print(f"{s}: json images={len(d['images'])} anns={len(d['annotations'])} files_on_disk={n_files} -> {'OK' if ok else 'MISMATCH'}")
json.dump(manifest, open(SAFE / "manifest.json", "w"), indent=2)

# remove a fake unmounted /content/drive stub left by the failed mount (local only)
if Path("/content/drive").exists() and not os.path.ismount("/content/drive"):
    shutil.rmtree("/content/drive", ignore_errors=True)
os.system("df -h /content | tail -1")
print("\nDATASET RESTORED AND VERIFIED" if all_ok else "\nSTOP: dataset check failed. Share this output.")
print(f"Elapsed: {time.time()-t0:.0f}s | ROOT = {ROOT}")

In [ ]:
import os, io, json, csv, time, hashlib, shutil, random
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

t0 = time.time()
ROOT = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
SPLITS = ["train", "val", "test"]
EXPECTED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}

try:
    tot, used, free = shutil.disk_usage("/content/drive/MyDrive")
    print(f"Drive: total {tot/1e9:.1f} GB | used {used/1e9:.1f} GB | free {free/1e9:.1f} GB")
except Exception as e:
    print("Drive usage unavailable:", e)

def pick_out():
    try:
        p = Path("/content/drive/MyDrive/marine-plastic-outputs/review"); p.mkdir(parents=True, exist_ok=True)
        (p / ".w").write_text("ok"); (p / ".w").unlink(); return p
    except Exception as e:
        print("Drive not writable -> using /content/outputs/review:", e)
        p = Path("/content/outputs/review"); p.mkdir(parents=True, exist_ok=True); return p
OUT = pick_out(); print("Outputs ->", OUT)

# ---------- load ----------
D = {}
for s in SPLITS:
    p = [q for q in sorted((ROOT / "annotation").glob("*.json")) if s in q.name.lower()][0]
    d = json.load(open(p))
    D[s] = {"images": d["images"], "anns": d["annotations"],
            "cats": {c["id"]: c["name"] for c in d["categories"]}}
    assert (len(d["images"]), len(d["annotations"])) == EXPECTED[s], f"count mismatch {s}"
CATS = D["train"]["cats"]
assert D["val"]["cats"] == CATS == D["test"]["cats"], "category mismatch across splits"
NAMES = [CATS[k] for k in sorted(CATS)]
print("Categories (%d):" % len(NAMES), NAMES)

def resolve(s, fn):
    for p in (ROOT / "images" / s / fn, ROOT / "images" / s / Path(fn).name):
        if p.exists(): return p
    return None

# ---------- per-split sanity + stats ----------
REP = {"categories": NAMES, "splits": {}}
CLS = {s: Counter() for s in SPLITS}
REL = {}; PERIMG = {}
for s in SPLITS:
    imgs = {i["id"]: i for i in D[s]["images"]}
    anns = D[s]["anns"]
    missing = [i["file_name"] for i in imgs.values() if resolve(s, i["file_name"]) is None]
    orphan = sum(a["image_id"] not in imgs for a in anns)
    per_img = Counter(a["image_id"] for a in anns)
    empty = sum(1 for i in imgs if per_img.get(i, 0) == 0)
    w, h, rel = [], [], []
    degenerate = oob = crowd = noseg = 0
    for a in anns:
        CLS[s][CATS[a["category_id"]]] += 1
        im = imgs.get(a["image_id"])
        if im is None: continue
        x, y, bw, bh = a["bbox"]
        if bw <= 0 or bh <= 0: degenerate += 1; continue
        if x < -1 or y < -1 or x + bw > im["width"] + 1 or y + bh > im["height"] + 1: oob += 1
        crowd += int(a.get("iscrowd", 0) == 1)
        noseg += int(not a.get("segmentation"))
        w.append(bw); h.append(bh); rel.append(bw * bh / (im["width"] * im["height"]))
    w, h, rel = np.array(w), np.array(h), np.array(rel); area = w * h
    sizes = Counter((i["width"], i["height"]) for i in imgs.values())
    cnts = np.array([per_img.get(i, 0) for i in imgs])
    REL[s] = rel; PERIMG[s] = cnts
    REP["splits"][s] = {
        "images": len(imgs), "annotations": len(anns), "missing_files": len(missing), "orphan_anns": orphan,
        "images_without_anns": empty, "degenerate_boxes": degenerate, "out_of_bounds_boxes": oob,
        "iscrowd": crowd, "no_segmentation": noseg,
        "objs_per_img_median": float(np.median(cnts)), "objs_per_img_max": int(cnts.max()),
        "box_w_median_px": float(np.median(w)), "box_h_median_px": float(np.median(h)),
        "box_rel_area_median": float(np.median(rel)), "box_rel_area_p10": float(np.percentile(rel, 10)),
        "frac_small_lt32sq": float((area < 32**2).mean()), "frac_medium": float(((area >= 32**2) & (area < 96**2)).mean()),
        "frac_large_ge96sq": float((area >= 96**2).mean()), "frac_side_lt16px": float(((w < 16) | (h < 16)).mean()),
        "image_sizes_top5": [(f"{k[0]}x{k[1]}", v) for k, v in sizes.most_common(5)], "n_distinct_sizes": len(sizes)}
    r = REP["splits"][s]
    print(f"\n[{s}] imgs={r['images']} anns={r['annotations']} missing_files={r['missing_files']} orphan={r['orphan_anns']} "
          f"empty_imgs={r['images_without_anns']} degenerate={r['degenerate_boxes']} oob={r['out_of_bounds_boxes']} "
          f"crowd={r['iscrowd']} no_seg={r['no_segmentation']}")
    print(f"     objs/img median={r['objs_per_img_median']:.0f} max={r['objs_per_img_max']} | box median {r['box_w_median_px']:.0f}x{r['box_h_median_px']:.0f}px "
          f"| rel-area median={r['box_rel_area_median']*100:.3f}% p10={r['box_rel_area_p10']*100:.4f}%")
    print(f"     COCO size split: small={r['frac_small_lt32sq']*100:.1f}% medium={r['frac_medium']*100:.1f}% large={r['frac_large_ge96sq']*100:.1f}% | side<16px={r['frac_side_lt16px']*100:.1f}%")
    print(f"     image sizes (top5): {r['image_sizes_top5']} | distinct={r['n_distinct_sizes']}")

print("\nCLASS COUNTS (train / val / test):")
tot_cls = Counter()
for n in NAMES:
    tot_cls[n] = sum(CLS[s][n] for s in SPLITS)
for n, _ in tot_cls.most_common():
    print(f"  {n:20s} {CLS['train'][n]:6d} {CLS['val'][n]:6d} {CLS['test'][n]:6d}  total={tot_cls[n]}")
vals = [v for v in tot_cls.values() if v > 0]
print(f"Imbalance ratio (max/min class): {max(vals)/min(vals):.1f}:1")
REP["class_counts"] = {s: dict(CLS[s]) for s in SPLITS}
with open(OUT / "class_distribution.csv", "w", newline="") as f:
    wr = csv.writer(f); wr.writerow(["class", "train", "val", "test", "total"])
    for n, _ in tot_cls.most_common(): wr.writerow([n, CLS["train"][n], CLS["val"][n], CLS["test"][n], tot_cls[n]])

# ---------- leakage audit: exact + near-duplicate hashing ----------
TABLE = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def proc(item):
    s, iid, p = item
    b = p.read_bytes()
    md5 = hashlib.md5(b).hexdigest()
    im = Image.open(io.BytesIO(b)); im.draft("L", (256, 256))
    im = im.convert("L").resize((9, 8), Image.LANCZOS)
    a = np.asarray(im, dtype=np.int16)
    return s, iid, p.name, md5, np.packbits((a[:, 1:] > a[:, :-1]).flatten())

items = []
for s in SPLITS:
    for i in D[s]["images"]:
        items.append((s, i["id"], resolve(s, i["file_name"])))
print(f"\nHashing {len(items)} images ...")
H = {s: {"names": [], "md5": [], "dh": []} for s in SPLITS}
with ThreadPoolExecutor(8) as ex:
    for s, iid, name, md5, dh in ex.map(proc, items):
        H[s]["names"].append(name); H[s]["md5"].append(md5); H[s]["dh"].append(dh)
for s in SPLITS: H[s]["dh"] = np.stack(H[s]["dh"])
print(f"Hashing done in {time.time()-t0:.0f}s total")

def mind(A, B):
    dist, idx = [], []
    for r in B:
        d = TABLE[np.bitwise_xor(A, r)].sum(axis=1)
        j = int(d.argmin()); dist.append(int(d[j])); idx.append(j)
    return np.array(dist), np.array(idx)

REP["leakage"] = {}
print("\nLEAKAGE AUDIT")
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    name_ov = len(set(H[a]["names"]) & set(H[b]["names"]))
    md5_ov = len(set(H[a]["md5"]) & set(H[b]["md5"]))
    dist, idx = mind(H[a]["dh"], H[b]["dh"])
    r = {"filename_overlap": name_ov, "exact_md5_overlap": md5_ov, "dhash_dist0": int((dist == 0).sum()),
         "dhash_le5": int((dist <= 5).sum()), "dhash_le10": int((dist <= 10).sum()), "queries": len(dist)}
    REP["leakage"][f"{a}->{b}"] = r
    print(f"  {a:5s}->{b:5s}: filename_overlap={name_ov} md5_overlap={md5_ov} | {b} imgs with a {a} match: dist0={r['dhash_dist0']} <=5={r['dhash_le5']} <=10={r['dhash_le10']} (of {len(dist)})")
    for q in np.argsort(dist)[:3]:
        print(f"      closest: {b}/{H[b]['names'][q]} ~ {a}/{H[a]['names'][idx[q]]} (hamming {dist[q]})")

# ---------- frozen split manifest ----------
manifest = {s: [{"file": n, "md5": m} for n, m in zip(H[s]["names"], H[s]["md5"])] for s in SPLITS}
json.dump(manifest, open(OUT / "split_manifest.json", "w"))
json.dump(REP, open(OUT / "review_report.json", "w"), indent=2, default=str)

# ---------- figures ----------
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
order = [n for n, _ in tot_cls.most_common()]
x = np.arange(len(order)); wd = 0.27
for k, s in enumerate(SPLITS):
    ax[0].bar(x + (k - 1) * wd, [max(CLS[s][n], 0.8) for n in order], wd, label=s)
ax[0].set_yscale("log"); ax[0].set_xticks(x); ax[0].set_xticklabels(order, rotation=60, ha="right"); ax[0].set_title("Class counts (log)"); ax[0].legend()
for s in SPLITS: ax[1].hist(np.log10(REL[s]), bins=60, alpha=0.5, label=s)
ax[1].set_title("log10 (box area / image area)"); ax[1].legend()
for s in SPLITS: ax[2].hist(PERIMG[s], bins=50, alpha=0.5, label=s)
ax[2].set_title("Objects per image"); ax[2].legend()
plt.tight_layout(); plt.savefig(OUT / "dataset_stats.png", dpi=130); plt.show()

random.seed(0)
imgs_t = {i["id"]: i for i in D["test"]["images"]}
by_img = {}
for a in D["test"]["anns"]: by_img.setdefault(a["image_id"], []).append(a)
pick = random.sample(list(imgs_t), 6)
fig, axs = plt.subplots(2, 3, figsize=(18, 10))
for axx, iid in zip(axs.ravel(), pick):
    im = imgs_t[iid]; axx.imshow(Image.open(resolve("test", im["file_name"])).convert("RGB"))
    for a in by_img.get(iid, []):
        x0, y0, bw, bh = a["bbox"]
        axx.add_patch(mpatches.Rectangle((x0, y0), bw, bh, fill=False, edgecolor="red", linewidth=0.8))
    axx.set_title(f"{im['file_name']} | {len(by_img.get(iid, []))} objs", fontsize=9); axx.axis("off")
plt.tight_layout(); plt.savefig(OUT / "test_samples_boxes.png", dpi=110); plt.show()

print(f"\nSaved to {OUT}: review_report.json, class_distribution.csv, split_manifest.json, dataset_stats.png, test_samples_boxes.png")
print(f"Total runtime: {time.time()-t0:.0f}s")

In [ ]:
import os, io, json, time, shutil, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

t0 = time.time()
ROOT = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
SPLITS = ["train", "val", "test"]
OUT = Path("/content/outputs/review"); OUT.mkdir(parents=True, exist_ok=True)

STUB = Path("/content/drive")
if STUB.exists() and not os.path.ismount(str(STUB)):
    for f in STUB.rglob("*"):
        if f.is_file(): shutil.copy(f, OUT / f.name)
    shutil.rmtree(STUB, ignore_errors=True)
print("Drive mounted:", os.path.ismount("/content/drive"), "| files in OUT:", sorted(p.name for p in OUT.iterdir()))

D = {}
for s in SPLITS:
    p = [q for q in sorted((ROOT / "annotation").glob("*.json")) if s in q.name.lower()][0]
    D[s] = json.load(open(p))["images"]

def resolve(s, fn):
    for p in (ROOT / "images" / s / fn, ROOT / "images" / s / Path(fn).name):
        if p.exists(): return p
    raise FileNotFoundError(fn)

TABLE = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def zs(v):
    v = v.astype(np.float32).ravel(); v = v - v.mean(); return v / (np.linalg.norm(v) + 1e-6)

def proc(item):
    s, p = item
    im = Image.open(p).convert("L")
    a = np.asarray(im.resize((9, 8), Image.LANCZOS), dtype=np.int16)
    g = zs(np.asarray(im.resize((24, 24), Image.LANCZOS)))
    return s, p.name, np.packbits((a[:, 1:] > a[:, :-1]).flatten()), g

items = [(s, resolve(s, i["file_name"])) for s in SPLITS for i in D[s]]
H = {s: {"names": [], "dh": [], "g": []} for s in SPLITS}
print(f"Hashing {len(items)} images ...")
with ThreadPoolExecutor(8) as ex:
    for n, (s, name, dh, g) in enumerate(ex.map(proc, items), 1):
        H[s]["names"].append(name); H[s]["dh"].append(dh); H[s]["g"].append(g)
        if n % 1000 == 0: print(f"  {n}/{len(items)} ({time.time()-t0:.0f}s)")
for s in SPLITS:
    H[s]["dh"] = np.stack(H[s]["dh"]); H[s]["g"] = np.stack(H[s]["g"])

PAIRS = []
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    for qi, r in enumerate(H[b]["dh"]):
        d = TABLE[np.bitwise_xor(H[a]["dh"], r)].sum(axis=1); j = int(d.argmin())
        PAIRS.append(dict(ref_split=a, ref=H[a]["names"][j], q_split=b, q=H[b]["names"][qi],
                          dist=int(d[j]), corr=float(H[a]["g"][j] @ H[b]["g"][qi])))
json.dump(PAIRS, open(OUT / "near_dup_pairs.json", "w"), indent=1)

print("\nNEAR-DUPLICATE SUMMARY (query split vs reference split)")
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    P = [p for p in PAIRS if p["ref_split"] == a and p["q_split"] == b]
    print(f"  {a}->{b}: dist<=3: {sum(p['dist']<=3 for p in P)} | dist<=5: {sum(p['dist']<=5 for p in P)} | "
          f"dist<=5 & corr>=0.90: {sum(p['dist']<=5 and p['corr']>=0.90 for p in P)} | "
          f"dist<=5 & corr>=0.95: {sum(p['dist']<=5 and p['corr']>=0.95 for p in P)}")

random.seed(1)
sel = []
for a, b in [("train", "test"), ("train", "val"), ("val", "test")]:
    P = sorted([p for p in PAIRS if p["ref_split"] == a and p["q_split"] == b], key=lambda p: (p["dist"], -p["corr"]))
    sel += P[:4]
mid = [p for p in PAIRS if p["ref_split"] == "train" and p["q_split"] == "test" and 6 <= p["dist"] <= 10]
sel += random.sample(mid, min(3, len(mid)))

fig, axs = plt.subplots(5, 6, figsize=(24, 16))
for ax in axs.ravel(): ax.axis("off")
for k, p in enumerate(sel):
    r_, c_ = k // 3, 2 * (k % 3)
    for off, (sp, nm, tag) in enumerate([(p["q_split"], p["q"], "Q"), (p["ref_split"], p["ref"], "R")]):
        ax = axs[r_, c_ + off]
        ax.imshow(Image.open(resolve(sp, nm)).convert("RGB").resize((320, 240)))
        ax.set_title(f"{tag}: {sp}/{nm}" + (f"\nd={p['dist']} r={p['corr']:.2f}" if off == 0 else ""), fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "near_dup_pairs.png", dpi=90); plt.show()

shutil.make_archive("/content/review_outputs", "zip", str(OUT))
try:
    from google.colab import files
    files.download("/content/review_outputs.zip")
except Exception as e:
    print("Download skipped:", e)
print(f"Done in {time.time()-t0:.0f}s. Outputs in {OUT}")

In [ ]:
import os, io, json, time, shutil, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

t0 = time.time()
ROOT = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
SPLITS = ["train", "val", "test"]
OUT = Path("/content/outputs/review"); OUT.mkdir(parents=True, exist_ok=True)

STUB = Path("/content/drive")
if STUB.exists() and not os.path.ismount(str(STUB)):
    for f in STUB.rglob("*"):
        if f.is_file(): shutil.copy(f, OUT / f.name)
    shutil.rmtree(STUB, ignore_errors=True)
print("Drive mounted:", os.path.ismount("/content/drive"), "| files in OUT:", sorted(p.name for p in OUT.iterdir()))

D = {}
for s in SPLITS:
    p = [q for q in sorted((ROOT / "annotation").glob("*.json")) if s in q.name.lower()][0]
    D[s] = json.load(open(p))["images"]

def resolve(s, fn):
    for p in (ROOT / "images" / s / fn, ROOT / "images" / s / Path(fn).name):
        if p.exists(): return p
    raise FileNotFoundError(fn)

TABLE = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def zs(v):
    v = v.astype(np.float32).ravel(); v = v - v.mean(); return v / (np.linalg.norm(v) + 1e-6)

def proc(item):
    s, p = item
    im = Image.open(p).convert("L")
    a = np.asarray(im.resize((9, 8), Image.LANCZOS), dtype=np.int16)
    g = zs(np.asarray(im.resize((24, 24), Image.LANCZOS)))
    return s, p.name, np.packbits((a[:, 1:] > a[:, :-1]).flatten()), g

items = [(s, resolve(s, i["file_name"])) for s in SPLITS for i in D[s]]
H = {s: {"names": [], "dh": [], "g": []} for s in SPLITS}
print(f"Hashing {len(items)} images ...")
with ThreadPoolExecutor(8) as ex:
    for n, (s, name, dh, g) in enumerate(ex.map(proc, items), 1):
        H[s]["names"].append(name); H[s]["dh"].append(dh); H[s]["g"].append(g)
        if n % 1000 == 0: print(f"  {n}/{len(items)} ({time.time()-t0:.0f}s)")
for s in SPLITS:
    H[s]["dh"] = np.stack(H[s]["dh"]); H[s]["g"] = np.stack(H[s]["g"])

PAIRS = []
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    for qi, r in enumerate(H[b]["dh"]):
        d = TABLE[np.bitwise_xor(H[a]["dh"], r)].sum(axis=1); j = int(d.argmin())
        PAIRS.append(dict(ref_split=a, ref=H[a]["names"][j], q_split=b, q=H[b]["names"][qi],
                          dist=int(d[j]), corr=float(H[a]["g"][j] @ H[b]["g"][qi])))
json.dump(PAIRS, open(OUT / "near_dup_pairs.json", "w"), indent=1)

print("\nNEAR-DUPLICATE SUMMARY (query split vs reference split)")
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    P = [p for p in PAIRS if p["ref_split"] == a and p["q_split"] == b]
    print(f"  {a}->{b}: dist<=3: {sum(p['dist']<=3 for p in P)} | dist<=5: {sum(p['dist']<=5 for p in P)} | "
          f"dist<=5 & corr>=0.90: {sum(p['dist']<=5 and p['corr']>=0.90 for p in P)} | "
          f"dist<=5 & corr>=0.95: {sum(p['dist']<=5 and p['corr']>=0.95 for p in P)}")

random.seed(1)
sel = []
for a, b in [("train", "test"), ("train", "val"), ("val", "test")]:
    P = sorted([p for p in PAIRS if p["ref_split"] == a and p["q_split"] == b], key=lambda p: (p["dist"], -p["corr"]))
    sel += P[:4]
mid = [p for p in PAIRS if p["ref_split"] == "train" and p["q_split"] == "test" and 6 <= p["dist"] <= 10]
sel += random.sample(mid, min(3, len(mid)))

fig, axs = plt.subplots(5, 6, figsize=(24, 16))
for ax in axs.ravel(): ax.axis("off")
for k, p in enumerate(sel):
    r_, c_ = k // 3, 2 * (k % 3)
    for off, (sp, nm, tag) in enumerate([(p["q_split"], p["q"], "Q"), (p["ref_split"], p["ref"], "R")]):
        ax = axs[r_, c_ + off]
        ax.imshow(Image.open(resolve(sp, nm)).convert("RGB").resize((320, 240)))
        ax.set_title(f"{tag}: {sp}/{nm}" + (f"\nd={p['dist']} r={p['corr']:.2f}" if off == 0 else ""), fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "near_dup_pairs.png", dpi=90); plt.show()

shutil.make_archive("/content/review_outputs", "zip", str(OUT))
try:
    from google.colab import files
    files.download("/content/review_outputs.zip")
except Exception as e:
    print("Download skipped:", e)
print(f"Done in {time.time()-t0:.0f}s. Outputs in {OUT}")

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

print("GPU:", subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip() or "NO GPU IN THIS RUNTIME")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())

ROOT = Path("/content/bepli_v2/raw/BePLi_dataset_v2/plastic_coco")
YOLO = Path("/content/yolo_bepli")
EXPECTED = {"train": (2234, 71889), "val": (744, 24643), "test": (744, 22040)}

cats = None; counts = {}
for s, (ni, na) in EXPECTED.items():
    p = [q for q in sorted((ROOT / "annotation").glob("*.json")) if s in q.name.lower()][0]
    d = json.load(open(p))
    if cats is None:
        cats = sorted(d["categories"], key=lambda c: c["id"])
        cmap = {c["id"]: i for i, c in enumerate(cats)}
    (YOLO / "images" / s).mkdir(parents=True, exist_ok=True)
    (YOLO / "labels" / s).mkdir(parents=True, exist_ok=True)
    by = {}
    for a in d["annotations"]: by.setdefault(a["image_id"], []).append(a)
    n_lines = 0
    for im in d["images"]:
        fn = Path(im["file_name"]).name
        src = ROOT / "images" / s / fn
        if not src.exists(): src = ROOT / "images" / s / im["file_name"]
        dst = YOLO / "images" / s / fn
        if not dst.exists():
            try: os.link(src, dst)
            except OSError: shutil.copy(src, dst)
        W, H = im["width"], im["height"]; lines = []
        for a in by.get(im["id"], []):
            x, y, w, h = a["bbox"]
            x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x1 <= x0 or y1 <= y0: continue
            lines.append(f"{cmap[a['category_id']]} {(x0+x1)/2/W:.6f} {(y0+y1)/2/H:.6f} {(x1-x0)/W:.6f} {(y1-y0)/H:.6f}")
        (YOLO / "labels" / s / (Path(fn).stem + ".txt")).write_text("\n".join(lines))
        n_lines += len(lines)
    counts[s] = (len(d["images"]), n_lines)
    print(f"{s}: images={len(d['images'])} label_lines={n_lines} expected={na} -> {'OK' if n_lines == na else 'MISMATCH'}")

with open(YOLO / "bepli.yaml", "w") as f:
    f.write(f"path: {YOLO}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n")
    for i, c in enumerate(cats): f.write(f"  {i}: {c['name']}\n")
ok = all(counts[s][1] == EXPECTED[s][1] and counts[s][0] == EXPECTED[s][0] for s in EXPECTED)
print("\nYOLO DATASET OK" if ok else "\nSTOP: mismatch, share this output")
print("YAML:", YOLO / "bepli.yaml")
os.system("df -h /content | tail -1")

In [ ]:
import os, shutil, time
from pathlib import Path

stamp = time.strftime("%Y%m%d_%H%M")
stage = Path("/content/backup_stage"); shutil.rmtree(stage, ignore_errors=True); stage.mkdir()
for src in ["/content/outputs", "/content/bepli_safe"]:
    if Path(src).exists():
        shutil.copytree(src, stage / Path(src).name)
for extra in ["/content/yolo_bepli/bepli.yaml"]:
    if Path(extra).exists():
        shutil.copy(extra, stage / Path(extra).name)

zip_path = shutil.make_archive(f"/content/marine_backup_{stamp}", "zip", str(stage))
print("Backup:", zip_path, f"{os.path.getsize(zip_path)/1e6:.1f} MB")
for p in sorted(stage.rglob("*")):
    if p.is_file(): print("  ", p.relative_to(stage), f"{p.stat().st_size/1e3:.0f} KB")

from google.colab import files
files.download(zip_path)